# Technical Note P9: SciPy Nonlinear Equations and Root Finding

**Wook Kang · Revised English edition · Version 1.0.0 · 3 October 2026**

This note connects a mathematical task to a numerical object, an implementation, and evidence about the result. It is an instructional computational document. Demonstration parameters are illustrative unless explicitly stated otherwise.

Run from a fresh kernel in cell order. See [the series README](../README.md) for setup and [editorial and verification notes](../docs/EDITORIAL_NOTES.md) for scope and limitations. The English prose has been reorganized and expanded from the supplied Korean notebook; this is an adapted edition rather than a line-by-line translation.

**Reading question:** What is given, what is unknown, what operation relates them, and what independent check can assess the output?

## Core mathematical structure

$$
Ax=b
$$

$$
\boxed{
F(x)=0
}
$$

$$
f(x)=0
$$

$$
\mathbf F(\mathbf x)=\mathbf 0
$$

In [1]:
# Reproducible display and scale-aware comparison.
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
def scaled_allclose(a, b, rtol=1e-12, atol=None, **kwargs):
    """Compare numerical examples using an absolute tolerance scaled to b."""
    a, b = np.asarray(a), np.asarray(b)
    scale = float(np.max(np.abs(b))) if b.size else 0.0
    if atol is None:
        atol = 1e-14 * max(scale, np.finfo(float).tiny)
    return np.allclose(a, b, rtol=rtol, atol=atol, **kwargs)


## 1. A linear equation is also a root problem

Rewrite Ax=b as Ax-b=0. General nonlinear solvers use the same residual idea but cannot exploit a fixed linear operator automatically.

$$
ax-b=0
$$

$$
f(x)=0
$$

In [2]:
from scipy import optimize
import numpy as np

def f(x):
    return 3*x - 6

sol = optimize.root_scalar(f, bracket=[0, 5])

print("root =", sol.root)
print("f(root) =", f(sol.root))

root = 2.0
f(root) = 0.0


## 2. Bracketed and open methods

A continuous scalar function with opposite signs at bracket endpoints has at least one root inside. Open methods use local information and need not preserve a bracket.

$$
f(a)f(b)<0
$$

## 3. Bisection

Bisection halves a valid sign-changing bracket. It is reliable under continuity but does not find every root or detect an even-multiplicity root through sign change alone.

$$
[a,b]
$$

In [3]:
def f(x):
    return x**3 - 2

sol = optimize.root_scalar(
    f,
    bracket=[1, 2],
    method="bisect",
    xtol=1e-12,
)

print("root =", sol.root)
print("iterations =", sol.iterations)
print("converged =", sol.converged)

root = 1.2599210498947286
iterations = 40
converged = True


## 4. Brent's method

Brent combines interpolation with bracket protection. Verify continuity and endpoint signs before interpreting a failed bracket as evidence about root existence.

In [4]:
sol = optimize.root_scalar(
    f,
    bracket=[1, 2],
    method="brentq",
    xtol=1e-12,
)

print("root =", sol.root)
print("iterations =", sol.iterations)
print("converged =", sol.converged)

root = 1.2599210498948537
iterations = 8
converged = True


## 5. Newton's method

Newton uses a local derivative to form a trial step. Near a simple root it can converge rapidly; zero derivatives and distant starting points can cause failure.

$$
x_{k+1}=
x_k-
\frac{f(x_k)}{f'(x_k)}
$$

In [5]:
def f(x):
    return x**3 - 2

def df(x):
    return 3*x**2

sol = optimize.root_scalar(
    f,
    x0=1.5,
    fprime=df,
    method="newton",
    xtol=1e-12,
)

print("root =", sol.root)
print("iterations =", sol.iterations)
print("converged =", sol.converged)

root = 1.2599210498948732
iterations = 5
converged = True


$$
\boxed{
\text{fast}
\neq
\text{globally robust}
}
$$

## 6. The secant method

The secant method approximates a slope from two iterates. It avoids an explicit derivative but can encounter a nearly zero denominator or leave the valid domain.

$$
x_{k+1}=
x_k-
f(x_k)
\frac{x_k-x_{k-1}}
{f(x_k)-f(x_{k-1})}
$$

In [6]:
sol = optimize.root_scalar(
    f,
    x0=1.0,
    x1=2.0,
    method="secant",
    xtol=1e-12,
)

print("root =", sol.root)
print("iterations =", sol.iterations)
print("converged =", sol.converged)

root = 1.2599210498948732
iterations = 8
converged = True


## 7. Read the result object

Inspect root, converged, flag, iterations, and function evaluations. A stopping flag is evidence about an algorithm's criterion rather than a proof of physical admissibility.

In [7]:
sol = optimize.root_scalar(f, bracket=[1,2], method="brentq")

print(sol)

      converged: True
           flag: converged
 function_calls: 9
     iterations: 8
           root: 1.2599210498948537
         method: brentq


## 8. Check the residual

Evaluate the original equation at the reported root. Compare a scaled residual and, when available, an independent reference solution.

$$
|f(x^*)|
$$

$$
\boxed{
\text{solver terminated}
\neq
\text{residual is automatically negligible}
}
$$

In [8]:
xstar = sol.root
print("|f(root)| =", abs(f(xstar)))

|f(root)| = 9.303668946358812e-14


## 9. Multiple roots

At a multiple root Newton's usual quadratic local convergence can be lost. Sign-changing brackets also miss roots where the function only touches zero.

$$
f(x)=x^2
$$

In [9]:
def f_mult(x):
    return x**2

def df_mult(x):
    return 2*x

sol = optimize.root_scalar(
    f_mult,
    x0=1.0,
    fprime=df_mult,
    method="newton",
    xtol=1e-12,
)

print("root =", sol.root)
print("iterations =", sol.iterations)

root = 9.094947017729282e-13
iterations = 40


## 10. Root existence and solver outcome

A failed algorithm does not prove that no root exists. Conversely, satisfying a numerical stopping criterion does not prove uniqueness.

$$
f(x)=x^2+1
$$

In [10]:
def no_real_root(x):
    return x**2 + 1

try:
    optimize.root_scalar(no_real_root, bracket=[-1, 1], method="brentq")
except ValueError as e:
    print("Expected failure:", e)

Expected failure: f(a) and f(b) must have different signs


## 11. Nonlinear systems

Supply a vector residual with one component per equation. A square root solve differs from minimizing an overdetermined residual.

$$
\mathbf F(\mathbf x)=\mathbf 0
$$

$$
F_1(x,y)=x^2+y^2-1
$$

$$
F_2(x,y)=x-y
$$

In [11]:
def F(z):
    x, y = z
    return np.array([
        x**2 + y**2 - 1,
        x - y,
    ])

sol = optimize.root(F, x0=[0.5, 0.5])

print("x* =", sol.x)
print("F(x*) =", F(sol.x))
print("success =", sol.success)
print("message =", sol.message)

x* = [0.70710678 0.70710678]
F(x*) = [-4.4408921e-16  0.0000000e+00]
success = True
message = The solution converged.


## 12. Residual vectors

Record the meaning and units of each component. A raw Euclidean norm can be dominated by equations with large numerical scales.

$$
\mathbf r=
\mathbf F(\mathbf x)
$$

$$
\|\mathbf F(\mathbf x)\|
$$

In [12]:
res_norm = np.linalg.norm(F(sol.x))

print("residual norm =", res_norm)

residual norm = 4.440892098500626e-16


## 13. Jacobian

The Jacobian maps a small state perturbation to a residual perturbation. Rows index equations and columns index unknowns.

$$
J_{ij}=
\frac{\partial F_i}{\partial x_j}
$$

$$
\mathbf J=
\begin{bmatrix}
2x & 2y\\
1 & -1
\end{bmatrix}
$$

In [13]:
def J(z):
    x, y = z
    return np.array([
        [2*x, 2*y],
        [1.0, -1.0],
    ])

print(J([0.5, 0.5]))

[[ 1.  1.]
 [ 1. -1.]]


## 14. Newton for systems

Solve J-delta-x=-F rather than explicitly inverting J. Reevaluate the original residual after accepting an update.

$$
J(\mathbf x_k)\Delta\mathbf x=
-\mathbf F(\mathbf x_k)
$$

$$
\mathbf x_{k+1}=
\mathbf x_k+\Delta\mathbf x
$$

$$
\boxed{
\text{nonlinear solve}
\rightarrow
\text{repeated linear solves}
}
$$

In [14]:
def newton_system(F, J, x0, tol=1e-10, maxiter=20):
    x = np.array(x0, dtype=float)

    for k in range(maxiter):
        r = F(x)
        if np.linalg.norm(r) < tol:
            return x, k, True

        dx = np.linalg.solve(J(x), -r)
        x = x + dx

    return x, maxiter, False

xstar, nit, ok = newton_system(F, J, [0.5, 0.5])

print("x* =", xstar)
print("iterations =", nit)
print("converged =", ok)
print("residual =", F(xstar))

x* = [0.70710678 0.70710678]
iterations = 4
converged = True
residual = [2.25530705e-12 0.00000000e+00]


## 15. Provide an analytic Jacobian

An analytic Jacobian can improve efficiency and reliability when correctly implemented. Check it against independent directional or finite-difference derivatives.

In [15]:
sol_j = optimize.root(F, x0=[0.5, 0.5], jac=J)

print("x* =", sol_j.x)
print("success =", sol_j.success)
print("nfev =", sol_j.nfev)
print("njev =", getattr(sol_j, "njev", None))

x* = [0.70710678 0.70710678]
success = True
nfev = 9
njev = 1


## 16. Numerical Jacobians

Finite differences approximate sensitivity using perturbed residual calls. Match perturbation size to parameter scale and forward-solver noise.

In [16]:
def numerical_jacobian(F, x, h=1e-6):
    x = np.asarray(x, dtype=float)
    f0 = np.asarray(F(x), dtype=float)
    Jnum = np.zeros((len(f0), len(x)))

    for j in range(len(x)):
        xp = x.copy()
        xm = x.copy()
        xp[j] += h
        xm[j] -= h
        Jnum[:, j] = (F(xp) - F(xm))/(2*h)

    return Jnum

x0 = np.array([0.5, 0.5])

print("analytic J =")
print(J(x0))
print("numerical J =")
print(numerical_jacobian(F, x0))

analytic J =
[[ 1.  1.]
 [ 1. -1.]]
numerical J =
[[ 1.  1.]
 [ 1. -1.]]


## 17. Derivative error and step size

A smaller step reduces truncation error only until cancellation and evaluation noise become significant. Test more than one step size.

$$
\boxed{
h\downarrow
\not\Rightarrow
\text{derivative accuracy always improves}
}
$$

## 18. State scaling

Variables with different units or magnitudes can distort step calculations. Nondimensionalization exposes a more interpretable numerical problem.

$$
x\sim10^{-10},
\qquad
y\sim10^5
$$

$$
\hat x=\frac{x}{x_{\mathrm{scale}}}
$$

## 19. Residual scaling

Scale equations by justified characteristic values or measurement uncertainties. Scaling changes numerical weighting and must be documented.

$$
F_1\sim10^{-8},
\qquad
F_2\sim10^3
$$

$$
\hat F_i=\frac{F_i}{F_{i,\mathrm{scale}}}
$$

$$
\boxed{
\text{variable scaling}
+
\text{residual scaling}
}
$$

## 20. Initial guesses and branches

Different initial guesses can reach different roots. Use physical constraints and continuation when selecting an admissible branch.

In [17]:
def F_multi(z):
    x, y = z
    return np.array([
        x**2 - 1,
        y**2 - 1,
    ])

for x0 in [[0.5,0.5], [-0.5,0.5], [0.5,-0.5], [-0.5,-0.5]]:
    sol = optimize.root(F_multi, x0=x0)
    print("x0 =", x0, "-> root =", sol.x)

x0 = [0.5, 0.5] -> root = [1. 1.]
x0 = [-0.5, 0.5] -> root = [-1.  1.]
x0 = [0.5, -0.5] -> root = [ 1. -1.]
x0 = [-0.5, -0.5] -> root = [-1. -1.]


## 21. Interpret success cautiously

A successful solver can return an inadmissible, nonunique, or poorly conditioned state. Check residuals, constraints, and sensitivity separately.

$$
\boxed{
\texttt{success=True}
\neq
\text{physical success}
}
$$

## 22. Singular Jacobians

A singular or nearly singular Jacobian can indicate redundant equations, a bifurcation, or poor scaling. A condition number helps diagnose local sensitivity.

$$
J\Delta x=-F
$$

$$
\kappa(J)
$$

In [18]:
Jstar = J(sol_j.x)

print("J at solution =")
print(Jstar)
print("condition number =", np.linalg.cond(Jstar))

J at solution =
[[ 1.41421356  1.41421356]
 [ 1.         -1.        ]]
condition number = 1.4142135623730947


## 23. Why damping is useful

A full Newton step can increase the residual or leave the model domain. Backtracking seeks an acceptable reduced step but must explicitly report failure if none is found.

$$
x_{k+1}=x_k+\Delta x
$$

$$
x_{k+1}=
x_k+\alpha\Delta x,
\qquad
0<\alpha\le1
$$

## 24. A damped Newton example

The revised teaching implementation checks finite values and derivatives, rejects an unsuccessful line search, and checks the final residual after the iteration limit.

In [19]:
def damped_newton_scalar(f, df, x0, tol=1e-10, maxiter=30, alpha_min=1e-6):
    """Teaching implementation: residual backtracking with explicit failure."""
    if tol <= 0 or maxiter < 1 or not 0 < alpha_min <= 1:
        raise ValueError("Invalid iteration settings.")
    x = float(x0)
    for k in range(maxiter):
        r = float(f(x))
        if not np.isfinite(r):
            return x, k, False
        if abs(r) <= tol:
            return x, k, True
        slope = float(df(x))
        if not np.isfinite(slope) or slope == 0:
            return x, k, False
        dx = -r / slope
        alpha = 1.0
        accepted = False
        while alpha >= alpha_min:
            trial = x + alpha * dx
            rt = float(f(trial))
            if np.isfinite(rt) and abs(rt) < abs(r):
                x, accepted = trial, True
                break
            alpha *= 0.5
        if not accepted:
            return x, k, False
    return x, maxiter, bool(np.isfinite(f(x)) and abs(f(x)) <= tol)

f = lambda x: x**3 - 2
df = lambda x: 3*x**2
xstar, nit, ok = damped_newton_scalar(f, df, 0.2)
print("root =", xstar)
print("iterations =", nit)
print("converged =", ok)
print("residual =", f(xstar))
assert ok and abs(f(xstar)) <= 1e-10
# A rootless stationary start must be reported as a failure.
assert not damped_newton_scalar(lambda x: x*x+1, lambda x: 2*x, 0)[2]


root = 1.2599210498948872
iterations = 4
converged = True
residual = 6.661338147750939e-14


## 25. Physical bounds

An unconstrained root method does not automatically enforce positivity or a bounded state. Use a justified parameterization or an appropriate constrained formulation.

## 26. Roots and least squares

A root seeks zero residual; least squares seeks the smallest residual norm. An inconsistent system can have a least-squares minimum with nonzero residual.

$$
F(x)=0
$$

$$
\min_\theta \|r(\theta)\|^2
$$

$$
\boxed{
\text{equation solving}
\neq
\text{parameter fitting}
}
$$

## 27. Nonlinear PDE residuals

After discretization, time, diffusion, sources, and boundaries contribute to one residual. Each contribution must use compatible units and unknown ordering.

$$
\mathbf R(\mathbf U)=0
$$

$$
R_i(U)=
U_i-U_i^n-
\Delta t\,L_i(U)
$$

## 28. Newton-PDE structure

Each Newton iteration builds or applies a Jacobian, solves for a correction, accepts a step, and reevaluates the residual. Linear and nonlinear tolerances have different roles.

$$
R(U)=0
$$

$$
J(U^k)\Delta U=
-R(U^k)
$$

$$
U^{k+1}=U^k+\Delta U
$$

$$
\boxed{
\text{nonlinear PDE}
\rightarrow
\text{residual}
\rightarrow
\text{Jacobian}
\rightarrow
\text{linear solve}
}
$$

## 29. Root-method map

Prefer a valid scalar bracket when available; choose derivatives and scaling deliberately for open or multivariable methods. Always check the original residual.

| Situation | Representative choice |  
| --- | --- |  
| Scalar with a valid bracket | `brentq` |  
| Scalar requiring bracket protection | bisection / Brent |  
| Scalar with a derivative | Newton |  
| Scalar without a derivative | secant |  
| multivariable square system | `optimize.root` |  
| noisy overdetermined fitting | `least_squares` |  
| Bounds required | Consider a constrained optimization formulation |

## 30. Ideas to retain

Separate existence, uniqueness, convergence, residual size, conditioning, and physical admissibility. A returned number answers none of these automatically.

$$
\boxed{
\texttt{success=True}
\neq
\text{physical validity}
}
$$

## 31. Continue to P10

Time integration advances a state according to a rate law and introduces a further distinction between local time-error control and global solution accuracy.

$$
\boxed{
F(x)=0
\rightarrow
\dot y=f(t,y)
}
$$

## Further work

1. Change one parameter or discretization choice and predict the effect before running.
2. Write the input and output shapes, units, and field locations.
3. Construct a deliberately invalid input and make the calculation report it clearly.
4. Separate an execution check from an accuracy check and from any physical claim.

## Primary documentation

- [SciPy root_scalar](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root_scalar.html)
- [SciPy least_squares](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html)

These are living documentation links. The accompanying requirements and verification report identify the versions actually executed for this edition.

## AI assistance and responsibility

The English adaptation, editorial supplementation, and code review were prepared with AI assistance. Wook Kang retains responsibility for reviewing the explanations, examples, and any subsequent research application. No new experimental validation is claimed.